In [1]:
import pandas as pd
import numpy as np

In [2]:
policy = pd.read_csv("../data/external/states_sports_betting_legalization.csv")
state_monthly = pd.read_csv("../data/processed/helpline_monthly.csv")
state_monthly.rename(columns={"caller_state": "State"}, inplace=True)
policy["Online Launch"] = pd.to_datetime(policy["Online Launch"])
policy["Retail Launch"] = pd.to_datetime(policy["Retail Launch"])
state_monthly["date"] = pd.to_datetime(state_monthly["date"])

In [3]:
# Remove dropped states from both policy and helpline panel
policy_clean = policy[policy['Treatment Group'] != 'Dropped'].copy()
state_monthly = state_monthly[state_monthly['State'].isin(policy_clean['State'])].copy().reset_index(drop=True)
state_monthly = state_monthly[
    (state_monthly['date'] >= '2016-01-01') &
    (state_monthly['date'] <= '2025-12-01')
].copy()
print(state_monthly['State'].nunique())  # should be 47

print(state_monthly.shape)              # should be 47 × 120 = 5,640 rows
print(state_monthly.duplicated(subset=['State', 'date']).sum())  # should be 0
state_monthly.rename(columns={'caller_state': 'State'}, inplace=True)
state_monthly = state_monthly.sort_values(['State', 'date']).reset_index(drop=True)
state_monthly

47
(5448, 5)
0


,State,year,month,total_contacts,date
0,Alabama,2016,1,864,2016-01-01
1,Alabama,2016,2,700,2016-02-01
2,Alabama,2016,3,619,2016-03-01
3,Alabama,2016,4,582,2016-04-01
4,Alabama,2016,5,622,2016-05-01
...,...,...,...,...,...
5443,Wyoming,2025,8,77,2025-08-01
5444,Wyoming,2025,9,58,2025-09-01
5445,Wyoming,2025,10,30,2025-10-01
5446,Wyoming,2025,11,21,2025-11-01


In [5]:
state_monthly['date'] = pd.to_datetime(state_monthly['date'])

full_index = pd.MultiIndex.from_product(
    [policy_clean['State'].unique(), pd.date_range('2016-01-01', '2025-12-01', freq='MS')],
    names=['State', 'date']
)

full_df = pd.DataFrame(index=full_index).reset_index()
missing = full_df.merge(state_monthly[['State', 'date']], on=['State', 'date'], how='left', indicator=True)
missing = missing[missing['_merge'] == 'left_only']

print(missing.shape)
print(missing.groupby('date').size().sort_values(ascending=False).to_string())
missing[missing["date"] == "2018-11-01"]

(192, 3)
date
2018-08-01    47
2018-12-01    47
2020-04-01    47
2020-08-01    47
2018-03-01     1
2018-04-01     1
2018-07-01     1
2018-11-01     1


,State,date,_merge
4474,South Dakota,2018-11-01,left_only


# Outages and Missing Values

In [7]:
# Drop South Dakota
state_monthly = state_monthly[state_monthly['State'] != 'South Dakota'].copy()
policy_clean = policy_clean[policy_clean['State'] != 'South Dakota'].copy()

# Reindex to full balanced panel (46 states × 120 months = 5,520)
full_index = pd.MultiIndex.from_product(
    [policy_clean['State'].unique(), pd.date_range('2016-01-01', '2025-12-01', freq='MS')],
    names=['State', 'date']
)
panel = pd.DataFrame(index=full_index).reset_index()
panel = panel.merge(state_monthly[['State', 'date', 'total_contacts']], on=['State', 'date'], how='left')

full_outages = ['2018-08-01', '2018-12-01', '2020-04-01', '2020-08-01']
partial_outages = ['2016-02-01', '2022-09-01']

panel['data_quality'] = 'ok'
panel.loc[panel['date'].isin(pd.to_datetime(full_outages)), 'data_quality'] = 'full_outage'
panel.loc[panel['date'].isin(pd.to_datetime(partial_outages)), 'data_quality'] = 'partial_outage'

panel = panel.sort_values(['State', 'date']).reset_index(drop=True)

panel['total_contacts'] = (
    panel.groupby('State')['total_contacts']
    .transform(lambda x: x.interpolate(method='linear', limit_area='inside'))
)

# Verify no NaNs remain
print(panel['total_contacts'].isna().sum())
panel
wv = panel[(panel['State'] == 'West Virginia') & (panel['date'].between('2018-06-01', '2018-10-01'))]
print(wv[['State', 'date', 'total_contacts', 'data_quality']])

0
              State       date  total_contacts data_quality
5189  West Virginia 2018-06-01            91.0           ok
5190  West Virginia 2018-07-01            93.0           ok
5191  West Virginia 2018-08-01            97.5  full_outage
5192  West Virginia 2018-09-01           102.0           ok
5193  West Virginia 2018-10-01           130.0           ok


# Treatment Variables

In [221]:
panel["treated"] = panel['online_launch'].notna().astype(int)
panel["post"] = (panel['date'] >= panel['online_launch']).astype(int)
panel["treat_post"] = panel["treated"] * panel["post"]
panel["online_only"] = ((panel['online_launch'].notna()) & (panel['retail_launch'].isna())).astype(int)
panel["time_to_treatment"] = panel.apply(
    lambda r: (r['date'].to_period('M') - r['online_launch'].to_period('M')).n 
    if pd.notna(r['online_launch']) else np.nan, axis=1
)

# Population

In [222]:
state_abbrev = {
    'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ', 'Arkansas': 'AR',
    'California': 'CA', 'Colorado': 'CO', 'Connecticut': 'CT', 'Delaware': 'DE',
    'Florida': 'FL', 'Georgia': 'GA', 'Hawaii': 'HI', 'Idaho': 'ID',
    'Illinois': 'IL', 'Indiana': 'IN', 'Iowa': 'IA', 'Kansas': 'KS',
    'Kentucky': 'KY', 'Louisiana': 'LA', 'Maine': 'ME', 'Maryland': 'MD',
    'Massachusetts': 'MA', 'Michigan': 'MI', 'Minnesota': 'MN', 'Mississippi': 'MS',
    'Missouri': 'MO', 'Montana': 'MT', 'Nebraska': 'NE', 'Nevada': 'NV',
    'New Hampshire': 'NH', 'New Jersey': 'NJ', 'New Mexico': 'NM', 'New York': 'NY',
    'North Carolina': 'NC', 'North Dakota': 'ND', 'Ohio': 'OH', 'Oklahoma': 'OK',
    'Oregon': 'OR', 'Pennsylvania': 'PA', 'Rhode Island': 'RI', 'South Carolina': 'SC',
    'South Dakota': 'SD', 'Tennessee': 'TN', 'Texas': 'TX', 'Utah': 'UT',
    'Vermont': 'VT', 'Virginia': 'VA', 'Washington': 'WA', 'West Virginia': 'WV',
    'Wisconsin': 'WI', 'Wyoming': 'WY'
}
panel["State"] = panel["State"].map(state_abbrev)


In [223]:
pop = pd.read_csv("../data/external/historical_state_population_by_year.csv", header=None, names=["State", "year", "Population"])
panel["year"] = panel['date'].dt.year
panel["month"] = panel['date'].dt.month

In [224]:
panel = panel.merge(pop, on=['State', 'year'], how='left')

In [225]:
panel['contacts_per_100k'] = panel['total_contacts'] / panel['Population'] * 100000

In [230]:
# assert len(panel) == 5640
# assert panel['total_contacts'].isna().sum() == 188  # 4 months x 47 states outage months
# assert panel['State'].nunique() == 47